# Decent-mobility CO2 budget maps — city x mode matrix

Rebuilds the budget-category choropleth maps (0-1 / 1-3 / 3-7 / 7+ kg CO2e per week)
directly from the saved per-user outputs of each city/mode notebook, instead of
re-running the full tour-building pipeline.

**Inputs expected** (already produced by each city notebook, e.g. `05_carbon_cost_of_decent_mobility/tours/decent_mobility_tours_turku.ipynb`):

```
./output/tour_{mode}_{city}_decent_per_user.parquet   # user_id, decent_mobility_co2
./output/tour_{mode}_{city}_typical_per_user.parquet  # user_id, home_gid9, ... (used only to attach home_gid9)
```

modes: `bike`, `pt`, `car`
cities: `helsinki`, `turku`, `tampere`, `oulu`

**Filter polygons** (city extent used to clip the hex grid before plotting):

```
./data/{city}_filter_map.geojson
```

**Grid layout:** 4 rows (cities) x 3 columns (modes), order:
- rows: Helsinki, Turku, Tampere, Oulu
- columns: bike, PT, car

If a file for a given city/mode is missing, that panel is rendered as a
"data not available" placeholder so the rest of the matrix still renders.

Two outputs are produced:
1. One big 4x3 composite figure (`output/matrix_city_mode.png`), high-DPI, for direct use in the paper.
2. One row-per-city PNG (`output/row_{city}.png`), each containing bike/PT/car side by side at full
   resolution with a shared colorbar, for cases where the rows are assembled separately in the final layout.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import h3
from shapely.geometry import Polygon
from pathlib import Path


## 1. Configuration

Paths and labels used throughout the notebook.

In [ ]:
# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------

CITIES = ["helsinki", "turku", "tampere", "oulu"]          # rows
CITY_LABELS = {"helsinki": "Helsinki", "turku": "Turku",
               "tampere": "Tampere", "oulu": "Oulu"}

MODES = ["bike", "pt", "car"]                               # columns
MODE_LABELS = {"bike": "Bike", "pt": "Public transport", "car": "Car"}

OUTPUT_DIR = Path("./output")
DATA_DIR = Path("./data")

# per-user decent-mobility CO2 file saved by each city notebook
# (only has user_id + decent_mobility_co2 -- no home hex)
def per_user_path(city, mode):
    return OUTPUT_DIR / f"tour_{mode}_{city}_decent_per_user.parquet"

# per-user "typical" file -- available for every city/mode, carries home_gid9.
# Used purely to attach each user's home hex onto the decent-mobility CO2 values above.
def typical_per_user_path(city, mode):
    return OUTPUT_DIR / f"tour_{mode}_{city}_typical_per_user.parquet"

# each city's filter/extent polygon
def filter_path(city):
    return DATA_DIR / f"{city}_filter_map.geojson"

# CO2 budget categories (grams CO2e / week) -- same bins as the original maps
BINS = [0, 1000, 3000, 7000, np.inf]
LABELS = ["0\u20131 kg", "1\u20133 kg", "3\u20137 kg", "7+ kg"]
COLORS = {
    "0\u20131 kg": "#08306b",   # very dark blue
    "1\u20133 kg": "#2171b5",   # medium blue
    "3\u20137 kg": "#9ecae1",   # light blue
    "7+ kg":   "#d73027",   # red - threshold violation
}

FIGS_DPI = 300


## 2. Helper functions

In [ ]:
def h3_to_poly(h):
    """H3 index -> shapely Polygon"""
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)
    return Polygon(boundary)


def load_city_mode_gdf(city, mode):
    """
    Load the per-user decent-mobility CO2 file for a city/mode. That file only
    has user_id + decent_mobility_co2 (no home hex), so home_gid9 is attached
    by merging on user_id with the corresponding typical_per_user file, which
    is available for every city/mode and carries home_gid9. Then aggregate to
    H3 hex level, build geometries, bin into budget categories, and clip to
    the city's filter polygon.

    Returns a GeoDataFrame (possibly empty) or None if either source file is missing.
    """
    fp = per_user_path(city, mode)
    fp_typical = typical_per_user_path(city, mode)
    if not fp.exists() or not fp_typical.exists():
        return None

    user_co2 = pd.read_parquet(fp)
    user_typical = pd.read_parquet(fp_typical)

    # be defensive about column naming -- some pipelines may save home_gid9
    # under a slightly different name
    home_col = "home_gid9" if "home_gid9" in user_typical.columns else None
    if home_col is None:
        for cand in ["home_id", "home_hex", "home_gid"]:
            if cand in user_typical.columns:
                home_col = cand
                break
    if home_col is None:
        raise ValueError(f"Could not find a home-hex column in {fp_typical}")

    home_lookup = user_typical[["user_id", home_col]].drop_duplicates("user_id")
    user_co2 = user_co2.merge(home_lookup, on="user_id", how="left")

    user_co2 = user_co2[user_co2[home_col].notna()]
    user_co2 = user_co2[(user_co2[home_col] != 1) & (user_co2[home_col] != "1")]

    hex_df = (
        user_co2
        .groupby(home_col, as_index=False)
        .agg(
            avg_co2=("decent_mobility_co2", "mean"),
            n_users=("user_id", "nunique"),
        )
        .rename(columns={home_col: "home_gid9"})
    )

    hex_df["geometry"] = hex_df["home_gid9"].apply(h3_to_poly)
    gdf = gpd.GeoDataFrame(hex_df, geometry="geometry", crs="EPSG:4326")
    gdf = gdf.to_crs(epsg=3857)

    gdf["co2_cat"] = pd.cut(
        gdf["avg_co2"], bins=BINS, labels=LABELS, include_lowest=True
    )

    # clip to city extent if a filter polygon is available
    fpoly = filter_path(city)
    if fpoly.exists():
        filt = gpd.read_file(fpoly).to_crs(epsg=3857)
        sel_geom = filt.union_all()
        gdf = gdf[gdf.within(sel_geom)]

    return gdf


def plot_panel(ax, gdf, city, mode, show_title=True):
    """Draw one city/mode hex map onto a given matplotlib axis."""
    if gdf is None:
        ax.text(
            0.5, 0.5, "data not available",
            ha="center", va="center", fontsize=9, color="gray",
            transform=ax.transAxes,
        )
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if show_title:
            ax.set_title(f"{CITY_LABELS[city]} \u2013 {MODE_LABELS[mode]}", fontsize=10)
        return

    if len(gdf) == 0:
        ax.text(
            0.5, 0.5, "no hexes in extent",
            ha="center", va="center", fontsize=9, color="gray",
            transform=ax.transAxes,
        )
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if show_title:
            ax.set_title(f"{CITY_LABELS[city]} \u2013 {MODE_LABELS[mode]}", fontsize=10)
        return

    for cat in LABELS:
        subset = gdf[gdf["co2_cat"] == cat]
        if len(subset) > 0:
            subset.plot(ax=ax, color=COLORS[cat], edgecolor="white", linewidth=0.1)

    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect("equal")

    if show_title:
        ax.set_title(f"{CITY_LABELS[city]} \u2013 {MODE_LABELS[mode]}", fontsize=10)


def shared_legend_handles():
    return [mpatches.Patch(color=COLORS[lab], label=lab) for lab in LABELS]


## 3. Load all city x mode hex layers once

Cached in a dict so the matrix and the per-city rows reuse the same data without re-reading parquet files.

In [ ]:
gdfs = {}
missing = []

for city in CITIES:
    for mode in MODES:
        g = load_city_mode_gdf(city, mode)
        gdfs[(city, mode)] = g
        if g is None:
            missing.append((city, mode))

if missing:
    print("Missing source files (will show as placeholders):")
    for city, mode in missing:
        print(f"  - {city} / {mode}: expected {per_user_path(city, mode)} and/or {typical_per_user_path(city, mode)}")
else:
    print("All 12 city/mode files loaded successfully.")


## 4. Full 4x3 matrix (rows = cities, columns = modes)

High-DPI single figure, suitable for direct use as a paper figure.

In [ ]:
n_rows, n_cols = len(CITIES), len(MODES)

fig = plt.figure(figsize=(4 * n_cols, 4 * n_rows))
gs = GridSpec(n_rows, n_cols, figure=fig, wspace=0.05, hspace=0.15)

for i, city in enumerate(CITIES):
    for j, mode in enumerate(MODES):
        ax = fig.add_subplot(gs[i, j])
        plot_panel(ax, gdfs[(city, mode)], city, mode, show_title=(i == 0))
        if j == 0:
            # row label (city name) on the left
            ax.text(
                -0.08, 0.5, CITY_LABELS[city],
                transform=ax.transAxes, rotation=90,
                ha="center", va="center", fontsize=12, fontweight="bold",
            )

# shared legend at the bottom
handles = shared_legend_handles()
fig.legend(
    handles=handles, loc="lower center", ncol=len(LABELS),
    bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=11,
    title="Weekly decent-mobility CO2e",
)

fig.suptitle("Decent-mobility CO2 budgets by city and mode", fontsize=15, y=1.0)

OUTPUT_DIR.mkdir(exist_ok=True)
matrix_path = OUTPUT_DIR / "matrix_city_mode.png"
fig.savefig(matrix_path, dpi=FIGS_DPI, bbox_inches="tight")
plt.show()

print(f"Saved: {matrix_path}")


## 5. Per-city rows (bike / PT / car), full resolution

For post-production: each row is saved as its own PNG at full resolution with a shared
colorbar/legend, sized and styled identically across cities so the 4 images align cleanly
when the rows are assembled separately in the final layout.

In [ ]:
import contextily as ctx; import basemaps


In [ ]:
ROW_FIGSIZE = (5 * len(MODES), 5)  # keep identical across all cities for alignment

for city in CITIES:
    fig, axes = plt.subplots(1, len(MODES), figsize=ROW_FIGSIZE)
    for j, mode in enumerate(MODES):
        plot_panel(axes[j], gdfs[(city, mode)], city, mode, show_title=True)
        if gdfs[(city, mode)] is not None and len(gdfs[(city, mode)]) > 0:
            ctx.add_basemap(axes[j], source=basemaps.POSITRON, attribution_size=4)

    handles = shared_legend_handles()
    fig.legend(
        handles=handles, loc="lower center", ncol=len(LABELS),
        bbox_to_anchor=(0.5, -0.07), frameon=False, fontsize=10,
        title="Weekly decent-mobility CO2e",
    )
    #fig.suptitle(CITY_LABELS[city], fontsize=14, y=1.05, fontweight="bold")
    row_path = OUTPUT_DIR / f"row_{city}.png"
    plt.tight_layout()
    fig.savefig(row_path, dpi=FIGS_DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {row_path}")

In [ ]:
"""
Hex-map grid of weekly decent-mobility CO2e by city and mode.
Nature-style aesthetics, using the project's saved bivariate palette
(diagonal slice as a sequential low->high CO2e ramp).
"""

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import contextily as ctx; import basemaps
from pathlib import Path

# ---------------------------------------------------------------
# Nature-style rcParams
# ---------------------------------------------------------------
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.linewidth": 0.6,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

LABELS = ["0\u20131 kg", "1\u20133 kg", "3\u20137 kg", "7+ kg"]

COLORS = {
    "0\u20131 kg": "#08306b",
    "1\u20133 kg": "#4385BE",
    "3\u20137 kg": "#9ecae1",
    "7+ kg":       "#3F005A",
}
FIGS_DPI = 300

# ---------------------------------------------------------------
# City boundary filters -- same pattern as gdf_home_income_filtered
# ---------------------------------------------------------------
city_filter_map = {
    "Tampere": "data/filter_hex_tampere.geojson",
    "Turku":   "data/filter_hex_turku.geojson",
    "Oulu":    "data/filter_hex_oulu.geojson",
}

city_boundaries = {
    city: gpd.read_file(path).to_crs("EPSG:3067").union_all()
    for city, path in city_filter_map.items()
}


def filter_hex_gdf(gdf, city):
    """Keep only hexes fully within the city's filter boundary.
    Helsinki (and any city not in city_boundaries) passes through unfiltered."""
    if gdf is None or len(gdf) == 0:
        return gdf
    if city not in city_boundaries:
        return gdf
    if gdf.crs != "EPSG:3067":
        gdf = gdf.to_crs("EPSG:3067")
    return gdf[gdf.geometry.within(city_boundaries[city])]


def plot_panel(ax, gdf, city, mode, show_title=True):
    """Draw one city/mode hex map onto a given matplotlib axis."""
    def _empty(msg):
        ax.text(
            0.5, 0.5, msg,
            ha="center", va="center", fontsize=8.5, color="#888888",
            transform=ax.transAxes,
        )
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if show_title:
            ax.set_title(f"{CITY_LABELS[city]} \u2013 {MODE_LABELS[mode]}",
                         fontsize=10, fontweight="bold", pad=6)

    if gdf is None:
        _empty("data not available")
        return
    if len(gdf) == 0:
        _empty("no hexes in extent")
        return

    for cat in LABELS:
        subset = gdf[gdf["co2_cat"] == cat]
        if len(subset) > 0:
            subset.plot(ax=ax, color=COLORS[cat], edgecolor="white",
                       linewidth=0.15, alpha=0.95)

    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect("equal")
    if show_title:
        ax.set_title(f"{CITY_LABELS[city]} \u2013 {MODE_LABELS[mode]}",
                     fontsize=10, fontweight="bold", pad=6)


def shared_legend_handles():
    return [mpatches.Patch(facecolor=COLORS[lab], edgecolor="#444444",
                           linewidth=0.4, label=lab) for lab in LABELS]


ROW_FIGSIZE = (5 * len(MODES), 5)

for city in CITIES:
    fig, axes = plt.subplots(1, len(MODES), figsize=ROW_FIGSIZE)

    for j, mode in enumerate(MODES):
        gdf_filtered = filter_hex_gdf(gdfs[(city, mode)], city)
        print(f"{city} - {mode}: "
              f"{len(gdfs[(city, mode)]) if gdfs[(city, mode)] is not None else 0} "
              f"-> {len(gdf_filtered) if gdf_filtered is not None else 0} after filter")
        plot_panel(axes[j], gdf_filtered, city, mode, show_title=True)
        if gdf_filtered is not None and len(gdf_filtered) > 0:
            ctx.add_basemap(axes[j], source=basemaps.POSITRON,
                           attribution_size=4)

    handles = shared_legend_handles()
    fig.legend(
        handles=handles, loc="lower center", ncol=len(LABELS),
        bbox_to_anchor=(0.5, -0.07), frameon=False, fontsize=9,
        title="Weekly decent-mobility CO$_2$e",
        title_fontsize=9.5,
    )

    row_path = OUTPUT_DIR / f"row_{city}.png"
    plt.tight_layout()
    fig.savefig(row_path, dpi=FIGS_DPI, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"row_{city}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {row_path}")

In [ ]:
oulu_filter = gpd.read_file("data/filter_hex_oulu.geojson")


In [ ]:
"""
Hex-map grid of weekly decent-mobility CO2e by city and mode.
Nature-style aesthetics, using the project's saved bivariate palette
(diagonal slice as a sequential low->high CO2e ramp).
"""

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import contextily as ctx; import basemaps
from pathlib import Path

# ---------------------------------------------------------------
# Nature-style rcParams
# ---------------------------------------------------------------
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.linewidth": 0.6,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

LABELS = ["0\u20131 kg", "1\u20133 kg", "3\u20137 kg", "7+ kg"]

COLORS = {
    "0\u20131 kg": "#08519C",   # clean navy blue, no purple tint
    "1\u20133 kg": "#4385BE",
    "3\u20137 kg": "#6BAED6",
    "7+ kg":       "#3F005A",
}
FIGS_DPI = 300

# ---------------------------------------------------------------
# City boundary filters -- same pattern as gdf_home_income_filtered
# ---------------------------------------------------------------
city_filter_map = {
    "Tampere": "data/filter_hex_tampere.geojson",
    "Turku":   "data/filter_hex_turku.geojson",
    "Oulu":    "data/filter_hex_oulu.geojson",
}

city_boundaries = {
    city: gpd.read_file(path).to_crs("EPSG:3067").union_all()
    for city, path in city_filter_map.items()
}


def filter_hex_gdf(gdf, city):
    """Keep only hexes fully within the city's filter boundary.
    Helsinki (and any city not in city_boundaries) passes through unfiltered."""
    if gdf is None or len(gdf) == 0:
        return gdf
    if city not in city_boundaries:
        return gdf
    if gdf.crs != "EPSG:3067":
        gdf = gdf.to_crs("EPSG:3067")
    return gdf[gdf.geometry.within(city_boundaries[city])]


def plot_panel(ax, gdf, city, mode, show_title=True):
    """Draw one city/mode hex map onto a given matplotlib axis."""
    def _empty(msg):
        ax.text(
            0.5, 0.5, msg,
            ha="center", va="center", fontsize=8.5, color="#888888",
            transform=ax.transAxes,
        )
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if show_title:
            ax.set_title(f"{CITY_LABELS[city]} \u2013 {MODE_LABELS[mode]}",
                         fontsize=10, fontweight="bold", pad=6)

    if gdf is None:
        _empty("data not available")
        return
    if len(gdf) == 0:
        _empty("no hexes in extent")
        return

    for cat in LABELS:
        subset = gdf[gdf["co2_cat"] == cat]
        if len(subset) > 0:
            subset.plot(ax=ax, color=COLORS[cat], edgecolor="white",
                       linewidth=0.15, alpha=0.95)

    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect("equal")
    if show_title:
        ax.set_title(f"{CITY_LABELS[city]} \u2013 {MODE_LABELS[mode]}",
                     fontsize=10, fontweight="bold", pad=6)


def shared_legend_handles():
    return [mpatches.Patch(facecolor=COLORS[lab], edgecolor="#444444",
                           linewidth=0.6, alpha=0.9, label=lab) for lab in LABELS]


ROW_FIGSIZE = (5 * len(MODES), 5)

for city in CITIES:
    fig, axes = plt.subplots(1, len(MODES), figsize=ROW_FIGSIZE)

    for j, mode in enumerate(MODES):
        gdf_filtered = filter_hex_gdf(gdfs[(city, mode)], city)
        print(f"{city} - {mode}: "
              f"{len(gdfs[(city, mode)]) if gdfs[(city, mode)] is not None else 0} "
              f"-> {len(gdf_filtered) if gdf_filtered is not None else 0} after filter")
        plot_panel(axes[j], gdf_filtered, city, mode, show_title=True)
        if gdf_filtered is not None and len(gdf_filtered) > 0:
            ctx.add_basemap(axes[j], source=basemaps.POSITRON,
                           attribution_size=4)

    handles = shared_legend_handles()
    fig.legend(
        handles=handles, loc="lower center", ncol=len(LABELS),
        bbox_to_anchor=(0.5, -0.09), frameon=False, fontsize=10,
        title="Weekly decent-mobility CO$_2$e",
        title_fontsize=10.5,
        handlelength=2.4,
        handleheight=2.4,
        columnspacing=1.8,
    )

    row_path = OUTPUT_DIR / f"row_{city}.png"
    plt.tight_layout()
    fig.savefig(row_path, dpi=FIGS_DPI, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"row_{city}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {row_path}")

## 6. Notes for post-production

- All panels use the same `BINS` / `COLORS` / `LABELS`, so colors are consistent across
  every city and mode -- safe to align/crop independently.
- `matrix_city_mode.png` is the single 12-panel composite (rows = cities, columns = modes).
- `row_helsinki.png`, `row_turku.png`, `row_tampere.png`, `row_oulu.png` are the same panels
  split one row per city, useful to:
  - align row heights and margins manually in the final layout, or
  - drop in a placeholder row once a missing city/mode file becomes available, without
    re-rendering the whole matrix.
- If a city/mode combination is missing, re-run this notebook after the corresponding
  city notebook has saved its `tour_{mode}_{city}_decent_per_user.parquet` AND
  `tour_{mode}_{city}_typical_per_user.parquet` files -- no other changes needed, the
  matrix will pick it up automatically.
